Day 1

Plot fake data dynamically

In [ ]:
import streamlit as st
import plotly.express as px
"""Plotly's Python API allows users to programmatically access Plotly's
server resources"""

st.title("Weather Forecast for the Next Days")
place = st.text_input("Place:")
days = st.slider("Forecat Days", min_value= 1, max_value= 5,
                help= "Select the number of forecasted days")
option = st.selectbox("Select data to view", "Temperature", "sky")
st.subheader(f"{option} for the next {days} days in {place}")

def get_data(days):
    date = []
    temp = []
    temp = [days * i for i in temp]
    return date, temp 

d, t = get_data(days)

figure = px.line(x = d, y = t, labels = {"x": "Date", "y": "Temperature (c)"})
st.plotly_chart(figure)

Day 2

Building the backend

In [ ]:
# backend
import requests

API_key = "your API"

def get_data(place, days, option):
    url = f"http://api.openweathermap.org/data/2.5/forecast?q={place}&appid={API_key}"
    requests = requests.get(url)
    content = requests.json()

if __name__ == "__main__":
    get_data()

Day 3

OpenWeatherMap API Client

In [ ]:
import requests

API_key = "your API"

def get_data(place, forecast_days, kind):
    url = f"http://api.openweathermap.org/data/2.5/forecast?q={place}&appid={API_key}"
    response = requests.get(url)
    data = response.json()
    filtered_data = data["list"]
    nr_values = 8 * forecast_days
    filtered_data = filtered_data[:nr_values]
    if kind == "Temperature":
        filtered_data = [item["main"]["temp"] for item in filtered_data]

    if kind == "Sky":
        filtered_data = [item["weather"][0]["main"] for item in filtered_data]
    return filtered_data


if __name__ == "__main__":
    get_data(place="Tehran", forecast_days=2, kind="Temperature")


Day 4



In [ ]:
import streamlit as st
import plotly.express as px
from backend import get_data

# Main title of the app
st.title("Weather Forecast for the Next Days")

# Text input for city name
place = st.text_input("Place:")

# Slider to select number of forecast days (1 to 5)
days = st.slider("Forecast Days", min_value=1, max_value=5,
                help="Select the number of forecasted days")

# Dropdown to choose data type (Temperature or Sky condition)
option = st.selectbox("Select data to view", ["Temperature", "Sky"])

# Display subtitle with user's selected options
st.subheader(f"{option} for the next {days} days in {place}")


if place:
    try:
        # Fetch weather data from backend API
        filtered_data = get_data(place, days)

        if option == "Temperature":
            # Extract temperature values (convert from Kelvin to Celsius)
            temperature = [item["main"]["temp"] / 10 for item in filtered_data]
            # Extract corresponding dates
            dates = [item["dt_txt"] for item in filtered_data]
            # Create interactive line chart using Plotly
            figure = px.line(x=dates, y=temperature, labels={"x": "Date", "y": "Temperature (c)"})
            st.plotly_chart(figure)

        if option == "Sky":
            # Dictionary mapping weather conditions to image file paths
            images = {
                "Clear": r"D:\porgram\code\mega corse\weather forecast data app\images\clear.png",
                "Rain": r"D:\porgram\code\mega corse\weather forecast data app\images\rain.png",
                "Clouds": r"D:\porgram\code\mega corse\weather forecast data app\images\cloud.png",
                "Snow": r"D:\porgram\code\mega corse\weather forecast data app\images\snow.png"
            }
            # Extract sky conditions from data
            sky_conditions = [item["weather"][0]["main"] for item in filtered_data]
            # Map conditions to image paths
            image_paths = [images[condition] for condition in sky_conditions]
            # Display icons in a row
            st.image(image_paths, width=115)

    # Handle invalid city name error
    except KeyError:
        st.write("That place doesn't exist")

In [ ]:
import requests

# OpenWeatherMap API key (sign up at https://openweathermap.org/api)
API_key = "your api"

def get_data(place, forecast_days):
    """
    Fetch weather forecast data from OpenWeatherMap API.
    
    Parameters:
    place (str): City name (e.g., "Tehran", "London")
    forecast_days (int): Number of forecast days (1 to 5)
    
    Returns:
    list: List of weather data for the specified number of days
          (8 records per day, each record = 3-hour interval)
    """
    
    # Build API request URL
    url = f"http://api.openweathermap.org/data/2.5/forecast?q={place}&appid={API_key}"
    
    # Send GET request to the API
    response = requests.get(url)
    
    # Parse JSON response into Python dictionary
    data = response.json()
    
    # Extract the forecast list (contains 40 records = 5 days × 8 records/day)
    filtered_data = data["list"]
    
    # Calculate total records needed (8 records per day)
    nr_values = 8 * forecast_days
    
    # Trim the list to only the requested number of days
    filtered_data = filtered_data[:nr_values]
    
    return filtered_data


if __name__ == "__main__":
    # Test with Tehran for 2 days
    result = get_data(place="Tehran", forecast_days=2)
    print(result)
